"""onyxia@vscode-python-gpu-336003-0:~/work$ cd DLforTimeSeries/
onyxia@vscode-python-gpu-336003-0:~/work/DLforTimeSeries$ cd moirai-classification/
onyxia@vscode-python-gpu-336003-0:~/work/DLforTimeSeries/moirai-classification$ uv sync
Resolved 146 packages in 1ms
Audited 141 packages in 1ms
onyxia@vscode-python-gpu-336003-0:~/work/DLforTimeSeries/moirai-classification$ uv run main.py"""
#puis chemin vers 3.11

In [21]:

import numpy as np

from tslearn.datasets import UCR_UEA_datasets
import torch
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from uni2ts.model.moirai import MoiraiModule
import torch.nn as nn
import torch.optim as optim

from encoder import MoiraiEncoder

# Data

Import

In [22]:
device = "cuda"
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

n_classes = len(set(y_train))
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

Prepare data

In [23]:
def preprocess_data(
    data: np.ndarray,
    *,
    device: str | torch.device = "cpu",
    dtype: torch.dtype = torch.float32,
):
    """
    data: np.ndarray of shape (N, T, V) = (n_individual, time, variate)
    Assumes NO missing values and NO padding in the raw data.
    """

    if not isinstance(data, np.ndarray):
        raise TypeError("data must be a numpy.ndarray")
    if data.ndim != 3:
        raise ValueError(f"Expected shape (N,T,V), got {data.shape}")

    N, T, V = data.shape

    # (N,T,V)
    past_target = torch.as_tensor(data, dtype=dtype, device=device)

    # observed mask: (N,T,V) all True no missing values
    past_observed_target = torch.ones((N, T, V), dtype=torch.bool, device=device)

    # padding mask: (N,T) all if no padding
    past_is_pad = torch.zeros((N, T), dtype=torch.bool, device=device)

    return past_target, past_observed_target, past_is_pad

In [24]:
X_train_target_tensor, X_train_is_target_observed, X_train_is_target_padded = (
    preprocess_data(X_train, device=device)
)

X_test_target_tensor, X_test_is_target_observed, X_test_is_target_padded = (
    preprocess_data(X_test, device=device)
)


print(X_train_target_tensor.shape)
print(X_train_is_target_observed.shape)
print(X_train_is_target_padded.shape)

torch.Size([2459, 36, 6])
torch.Size([2459, 36, 6])
torch.Size([2459, 36])


Creat Z_train and Z_test for each patch size

In [25]:
# Paramètres à tester
patch_sizes = [8, 16, 32, 64]

SIZE = "small"
device = "cuda"

Z_train_patch = {}
Z_test_patch = {}
for patch in patch_sizes:
    
    # 1. Définir les paramètres et l'encodeur pour la taille de patch actuelle
    MODEL_PARAMS = {
        "patch_size": patch,
        "num_samples": 100,
        "target_dim": 1,
        "feat_dynamic_real_dim": 0,
        "past_feat_dynamic_real_dim": 0,
    }
    
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=0,
        context_length=36,
        **MODEL_PARAMS,
    )
    encoder.eval()
    encoder.to(device)
    
    # 2. Extraire les embeddings (Z_train et Z_test)
    with torch.no_grad():
        Z_train = encoder(
            past_target=X_train_target_tensor,
            past_observed_target=X_train_is_target_observed,
            past_is_pad=X_train_is_target_padded,
        )
        Z_test = encoder(
            past_target=X_test_target_tensor,
            past_observed_target=X_test_is_target_observed,
            past_is_pad=X_test_is_target_padded,
        )
        
    Z_train_patch[patch] = Z_train
    Z_test_patch[patch] = Z_test

In [26]:

print(f"{'Patch Size':<12} | {'Train Shape':<25} | {'Test Shape':<25}")
for patch in patch_sizes:
    train_tensor = Z_train_patch[patch]
    test_tensor = Z_test_patch[patch]
    print(f"{patch:<12} | {str(list(train_tensor.shape)):<25} | {str(list(test_tensor.shape)):<25}")

Patch Size   | Train Shape               | Test Shape               
8            | [2459, 30, 384]           | [2466, 30, 384]          
16           | [2459, 18, 384]           | [2466, 18, 384]          
32           | [2459, 12, 384]           | [2466, 12, 384]          
64           | [2459, 6, 384]            | [2466, 6, 384]           


# Basic pooling with logistic regression

We started by applying simple pooling techniques (max, min, or mean) and comparing their performance across various patch sizes. Specifically, we evaluated whether to apply this pooling globally across all patches, locally within a single series, or across corresponding patches from all series.

## Training

Define one function for every type of pooling

In [27]:
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=device)

num_classes = len(torch.unique(y_train_tensor))
NUM_VARS = 6

def apply_pooling_pt(Z_tensor, method, num_vars=NUM_VARS):
    N, S, F = Z_tensor.shape
    P = S // num_vars # Calcul automatique du nombre de patches par variable
    
    # On reshape le tenseur pour séparer les Variables et les Patches
    # Forme résultante : (Batch, Variables, Patches, Features)
    Z_reshaped = Z_tensor.view(N, num_vars, P, F)
    
    # Basique et Global
    if method == "flatten":
        return Z_tensor.reshape(N, -1)
        
    elif method == "global_mean":
        return Z_tensor.mean(dim=1)
        
    elif method == "global_max":
        return Z_tensor.max(dim=1).values
        
    elif method == "global_min":
        return Z_tensor.min(dim=1).values
    
    elif method == "global_mean_max_min":
        return torch.cat([
            Z_tensor.mean(dim=1),
            Z_tensor.max(dim=1).values,
            Z_tensor.min(dim=1).values
        ], dim=1)

    # Pooling sur les Patches (on garde les variables distinctes) ---
    # Réduction sur la dimension 2 (Patches). Résultat : (N, num_vars, F), puis on aplatit
    elif method == "mean_over_patches":
        return Z_reshaped.mean(dim=2).reshape(N, -1)
        
    elif method == "max_over_patches":
        return Z_reshaped.max(dim=2).values.reshape(N, -1)
        
    elif method == "min_over_patches":
        return Z_reshaped.min(dim=2).values.reshape(N, -1)
        
    elif method == "mean_max_min_over_patches":
        p_mean = Z_reshaped.mean(dim=2).reshape(N, -1)
        p_max  = Z_reshaped.max(dim=2).values.reshape(N, -1)
        p_min  = Z_reshaped.min(dim=2).values.reshape(N, -1)
        return torch.cat([p_mean, p_max, p_min], dim=1)

    # Pooling sur les Variables (on synchronise les patches entre variables) ---
    # Réduction sur la dimension 1 (Variables). Résultat : (N, P, F), puis on aplatit
    elif method == "mean_over_variables":
        return Z_reshaped.mean(dim=1).reshape(N, -1)
        
    elif method == "max_over_variables":
        return Z_reshaped.max(dim=1).values.reshape(N, -1)
        
    elif method == "min_over_variables":
        return Z_reshaped.min(dim=1).values.reshape(N, -1)
        
    elif method == "mean_max_min_over_variables":
        v_mean = Z_reshaped.mean(dim=1).reshape(N, -1)
        v_max  = Z_reshaped.max(dim=1).values.reshape(N, -1)
        v_min  = Z_reshaped.min(dim=1).values.reshape(N, -1)
        return torch.cat([v_mean, v_max, v_min], dim=1)

    else:
        raise ValueError(f"Method {method} unknow")

Our machine have GPU so we do a Logistic Regression  with pytorch to be faster

In [28]:
class LogisticRegressionPT(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(LogisticRegressionPT, self).__init__()
        self.linear = nn.Linear(input_dim, num_classes)
        
    def forward(self, x):
        return self.linear(x)

We creeat a dataframe to stock the results and we fill it

In [29]:
pooling_methods = [
    "flatten", 
    "global_mean", "global_max", "global_min", "global_mean_max_min",
    "mean_over_patches", "max_over_patches", "min_over_patches", "mean_max_min_over_patches",
    "mean_over_variables", "max_over_variables", "min_over_variables", "mean_max_min_over_variables"
]

df_results = pd.DataFrame(index=pooling_methods, columns=patch_sizes)
df_results.index.name = "Pooling \\ Patch"

for patch in patch_sizes:
    Z_train = Z_train_patch[patch].to(device)
    Z_test = Z_test_patch[patch].to(device)
    
    for pool in pooling_methods:
        # Appliquer le pooling avec la nouvelle logique
        X_train_pool = apply_pooling_pt(Z_train, pool)
        X_test_pool = apply_pooling_pt(Z_test, pool)
        
        input_dim = X_train_pool.shape[1]
        
        # Initialiser le modèle
        model = LogisticRegressionPT(input_dim, num_classes).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.01)
        
        # Entraînement rapide
        epochs = 200
        model.train()
        for epoch in range(epochs):
            optimizer.zero_grad()
            outputs = model(X_train_pool)
            loss = criterion(outputs, y_train_tensor)
            loss.backward()
            optimizer.step()
            
        # Évaluation
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test_pool)
            _, predicted = torch.max(test_outputs.data, 1)
            
            correct = (predicted == y_test_tensor).sum().item()
            total = y_test_tensor.size(0)
            acc = correct / total
            
        # Remplissage discret du DataFrame
        df_results.loc[pool, patch] = round(acc, 2)

## Results

In [30]:
# Affichage propre du DataFrame
df_results = df_results.astype(float)
display(df_results)  # Remplace par print(df_results) si display n'est pas supporté

# Recherche de la meilleure combinaison
best_acc = df_results.max().max()
best_pool, best_patch = df_results.stack().idxmax()


print(f"MEILLEURE COMBINAISON : Patch = {best_patch} | Pooling = '{best_pool}'")
print(f"Précision maximale : {best_acc:.2f}")

,8,16,32,64
Pooling \ Patch,,,,
flatten,0.58,0.58,0.57,0.58
global_mean,0.59,0.59,0.57,0.58
global_max,0.55,0.54,0.53,0.57
global_min,0.54,0.55,0.53,0.56
global_mean_max_min,0.59,0.58,0.56,0.57
mean_over_patches,0.60,0.58,0.56,0.58
max_over_patches,0.60,0.58,0.56,0.58
min_over_patches,0.58,0.57,0.55,0.57
mean_max_min_over_patches,0.61,0.58,0.58,0.57


MEILLEURE COMBINAISON : Patch = 8 | Pooling = 'mean_max_min_over_patches'
Précision maximale : 0.61


We observe that smaller patch sizes of 8 yield better results by capturing more granular information, and that minimizing cross-variable aggregation by pooling over patches rather than series produces the highest accuracy.

# Advanced Pooling

## Generalized Mean (GeM) Pooling (Dont work yet)

Unlike standard methods (like mean or max), Generalized Mean (GeM) Pooling introduces a trainable parameter $p$ that the neural network learns during backpropagation. 
The formula is: 
$$f = \left(\frac{1}{N} \sum |x_i|^p\right)^\frac{1}{p}$$

To find the best approach, we will test GeM pooling across different dimensions dynamically during training:
- **Global**: Pooling over the entire sequence at once.
- **Over Patches**: Keeping variables separate and pooling their respective patches.
- **Over Variables**: Synchronizing patches and pooling across the different series.

For more details, you can refer to the original paper: [Fine-tuning CNN Image Retrieval with No Human Annotation (Radenović et al.)](https://arxiv.org/abs/1711.02512).

In [31]:
class GeMPooling(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super(GeMPooling, self).__init__()
        # The parameter p is learnable by the optimizer
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x, dim):
        # Clamp values to avoid NaN errors with fractional powers
        x = x.clamp(min=self.eps)
        return x.pow(self.p).mean(dim=dim).pow(1.0 / self.p)


class DynamicGeMClassifierPT(nn.Module):
    def __init__(self, num_vars, num_classes):
        super(DynamicGeMClassifierPT, self).__init__()
        self.num_vars = num_vars
        self.gem = GeMPooling(p=3.0)
        
        # LazyLinear automatically infers the input size during the first forward pass.
        # This allows us to dynamically change the pooling dimension without crashing!
        self.linear = nn.LazyLinear(num_classes)
        
    def forward(self, z, pool_dim):
        N, S, F = z.shape
        
        if pool_dim == "global":
            # Pool over the entire sequence dimension (dim=1)
            pooled = self.gem(z, dim=1)
            
        else:
            P = S // self.num_vars
            # Reshape: (Batch, Variables, Patches, Features)
            z_reshaped = z.view(N, self.num_vars, P, F)
            
            if pool_dim == "patches":
                # Pool over the Patches dimension (dim=2)
                pooled = self.gem(z_reshaped, dim=2)
            elif pool_dim == "variables":
                # Pool over the Variables dimension (dim=1)
                pooled = self.gem(z_reshaped, dim=1)
            else:
                raise ValueError("pool_dim must be 'global', 'patches', or 'variables'")
        
        # Flatten for the linear layer
        pooled_flat = pooled.view(N, -1)
        return self.linear(pooled_flat)

### Dynamic Training and Evaluation

We will now train the GeM model by iterating over both the `patch_sizes` and the `pooling_dimensions`. We will also monitor what value of $p$ the network naturally converged to for each configuration.

In [32]:
pooling_dimensions = ["global", "patches", "variables"]
df_gem_results = pd.DataFrame(index=pooling_dimensions, columns=patch_sizes)
df_gem_results.index.name = "GeM Pooling Dimension"

for patch in patch_sizes:
    Z_train = Z_train_patch[patch].to(device)
    Z_test = Z_test_patch[patch].to(device)
    
    for pool_dim in pooling_dimensions:
        # Instantiate the model
        model = DynamicGeMClassifierPT(num_vars=NUM_VARS, num_classes=num_classes).to(device)
        
        # IMPORTANT: We must perform a "dummy" forward pass with a tiny batch
        # to initialize the LazyLinear weights before creating the optimizer.
        _ = model(Z_train[:2], pool_dim=pool_dim)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.01)
        
        epochs = 200
        model.train()
        for epoch in range(epochs):
            optimizer.zero_grad()
            # We explicitly tell the model which dimension to pool over
            outputs = model(Z_train, pool_dim=pool_dim) 
            loss = criterion(outputs, y_train_tensor)
            loss.backward()
            optimizer.step()
            
        # Evaluation
        model.eval()
        with torch.no_grad():
            test_outputs = model(Z_test, pool_dim=pool_dim)
            _, predicted = torch.max(test_outputs.data, 1)
            
            correct = (predicted == y_test_tensor).sum().item()
            total = y_test_tensor.size(0)
            acc = correct / total
            
            # Retrieve the learned p value
            learned_p = model.gem.p.item()
            
        df_gem_results.loc[pool_dim, patch] = round(acc, 2)
        print(f"Patch: {patch:<4} | Dim: {pool_dim:<10} | Test Acc: {acc:.2f} | Learned p: {learned_p:.2f}")


display(df_gem_results.astype(float))

Patch: 8    | Dim: global     | Test Acc: 0.56 | Learned p: 4.28
Patch: 8    | Dim: patches    | Test Acc: 0.61 | Learned p: 3.73
Patch: 8    | Dim: variables  | Test Acc: 0.56 | Learned p: 4.00
Patch: 16   | Dim: global     | Test Acc: 0.55 | Learned p: 4.23
Patch: 16   | Dim: patches    | Test Acc: 0.59 | Learned p: 3.83
Patch: 16   | Dim: variables  | Test Acc: 0.56 | Learned p: 4.10
Patch: 32   | Dim: global     | Test Acc: 0.54 | Learned p: 4.37
Patch: 32   | Dim: patches    | Test Acc: 0.57 | Learned p: 4.18
Patch: 32   | Dim: variables  | Test Acc: 0.56 | Learned p: 4.45
Patch: 64   | Dim: global     | Test Acc: 0.56 | Learned p: 4.82
Patch: 64   | Dim: patches    | Test Acc: 0.59 | Learned p: 3.06
Patch: 64   | Dim: variables  | Test Acc: 0.56 | Learned p: 4.59


,8,16,32,64
GeM Pooling Dimension,,,,
global,0.56,0.55,0.54,0.56
patches,0.61,0.59,0.57,0.59
variables,0.56,0.56,0.56,0.56


TODO :
- target_dim?
- result diff de maxime car moi moi pytorch pour reglog